# 03 · Evaluation and error analysis
Fine-tuned model vs. the prompting baselines on the **full Spider dev set**: execution accuracy overall and by hardness,
a paired significance test, an automatic error taxonomy, and qualitative examples. Runtime: **T4 GPU**, about 15 min.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, time, warnings
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # less GPU memory fragmentation
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("T2S_SMOKE") == "1"      # local CPU smoke test (ignore on Colab/Kaggle)
ON_KAGGLE = os.path.exists("/kaggle/input")
try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
from pathlib import Path
if ON_KAGGLE:
    # Code is attached as a Kaggle Dataset (read-only) -> copy it to the writable /kaggle/working
    import glob, shutil, zipfile
    PROJECT_DIR = "/kaggle/working/text2sql"

    def _walk(top="/kaggle/input"):
        return os.walk(top, followlinks=True)   # Kaggle may mount inputs via symlinks

    def _version(code_root):
        import re as _re
        m = _re.search(r"__version__\s*=\s*(\d+)", (Path(code_root) / "src/t2s/__init__.py").read_text())
        return int(m.group(1)) if m else 0

    def _find_code():
        zips, found = [], []
        for root, dirs, files in _walk():
            if root.endswith("src/t2s") and "modeling.py" in files and not root.startswith(PROJECT_DIR):
                found.append(Path(root).parents[1])
        if found:   # several attached versions -> use the newest code
            found.sort(key=_version, reverse=True)
            if len(found) > 1:
                print("several code copies found:", [(str(f), _version(f)) for f in found])
            return found[0]
        for root, dirs, files in _walk():
            zips += [Path(root) / f for f in files if f.lower().endswith(".zip") and "text2sql" in f.lower()]
        for z in zips:   # dataset kept as an un-extracted zip -> extract it ourselves
            tmp = Path("/kaggle/tmp/code"); tmp.mkdir(parents=True, exist_ok=True)
            zipfile.ZipFile(z).extractall(tmp)
            for root, dirs, files in os.walk(tmp):
                if root.endswith("src/t2s") and "modeling.py" in files:
                    return Path(root).parents[1]
        return None

    code_root = _find_code()
    if code_root is None:
        print("Could not find the text2sql code. Contents of /kaggle/input:")
        for root, dirs, files in _walk():
            depth = root.count(os.sep) - 2
            if depth <= 4:
                print("  " * depth + os.path.basename(root) + "/", (files[:5] if files else ""))
        raise RuntimeError("Code dataset not attached. Right panel -> Input -> '+ Add Input' -> Your Work -> Datasets -> "
                           "text2sql-code -> Add. Then run again. (Send the listing above if it is attached.)")
    print("code found at", code_root, "| code version", _version(code_root))
    # Kaggle images ship torchao 0.10, which makes peft refuse to load LoRA adapters on fp16 models. We don't need it.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], capture_output=True)
    shutil.copytree(code_root, PROJECT_DIR, dirs_exist_ok=True,   # always refresh the code
                    ignore=shutil.ignore_patterns("artifacts", "results", "__pycache__"))
    for m in [m for m in list(sys.modules) if m == "t2s" or m.startswith("t2s.")]:
        del sys.modules[m]   # pick up new code if the kernel was not restarted
    # Resume support: if a previous run's Output is attached as an input, reuse its results/artifacts
    for root, dirs, files in _walk():
        if os.path.basename(root) == "results" and os.path.basename(os.path.dirname(root)) == "text2sql":
            prev_root = Path(root).parent
            for sub_dir in ["results", "artifacts", "reports"]:
                if (prev_root / sub_dir).exists():
                    shutil.copytree(prev_root / sub_dir, f"{PROJECT_DIR}/{sub_dir}", dirs_exist_ok=True)
            print("restored previous results from", prev_root)
    if not SMOKE:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
elif SMOKE:
    PROJECT_DIR = os.environ["T2S_PROJECT_DIR"]
elif not ON_COLAB:
    # Local machine (VS Code / Jupyter): the notebook lives in text2sql/notebooks/
    PROJECT_DIR = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/text2sql"   # <- change if needed
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART, FIG, RES = f"{PROJECT_DIR}/artifacts", f"{PROJECT_DIR}/reports/figures", f"{PROJECT_DIR}/results"
for d in (ART, FIG, RES): os.makedirs(d, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid"); plt.rcParams["figure.dpi"] = 110
from t2s.spider import download_spider, find_spider_root, load_split, db_path, serialize_schema, schema_stats
from t2s import modeling as MD, evaluate as EV
from t2s.analysis import hardness, HARDNESS_ORDER

# ---- data ----
if SMOKE:
    from t2s.smoke import make_fake_spider
    SPIDER = find_spider_root(make_fake_spider(f"{PROJECT_DIR}/data"))
else:
    if ON_KAGGLE:
        attached = [os.path.join(r, "dev.json") for r, d, f in os.walk("/kaggle/input", followlinks=True)
                    if "dev.json" in f and os.path.isdir(os.path.join(r, "database"))]   # Spider attached as input?
        SPIDER = find_spider_root(Path(attached[0]).parent) if attached else download_spider("/kaggle/working/spider")
    elif ON_COLAB:
        SPIDER = download_spider("/content/spider", drive_zip="/content/drive/MyDrive/masters-projects/spider_data.zip")
    else:
        SPIDER = download_spider(f"{PROJECT_DIR}/data/spider", drive_zip=f"{PROJECT_DIR}/data/spider_data.zip")
train_all, dev = load_split(SPIDER, "train"), load_split(SPIDER, "dev")

# ---- model ----
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
if SMOKE:
    from t2s.smoke import make_tiny_model
    tiny = f"{ART}/tiny-model"
    if not os.path.exists(f"{tiny}/config.json"):
        make_tiny_model(tiny, [e.question + " " + e.query for e in train_all] + [serialize_schema(db_path(SPIDER, e.db_id), "ddl_rows") for e in train_all[:20] + dev])
    BASE_MODEL = tiny
DEV_SUBSET = sorted(np.random.RandomState(0).choice(len(dev), min(300, len(dev)), replace=False))  # fixed subset for ablations
SEED = 42
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()
def save_json(o, name):
    json.dump(o, open(f"{ART}/{name}.json", "w"), indent=2, default=float)
def load_json(name, default=None):
    p = f"{ART}/{name}.json"; return json.load(open(p)) if os.path.exists(p) else default
def cached(name, fn):
    """Run fn() once and store its DataFrame as results/<name>.parquet; later calls (after a crash) just load it."""
    p = f"{RES}/{name}.parquet"
    if os.path.exists(p):
        df = pd.read_parquet(p); print(f"[cached] {name}: EX={df.exec_match.mean():.4f}"); return df
    df = fn(); df.to_parquet(p, index=False); print(f"[done] {name}: EX={df.exec_match.mean():.4f}"); return df
def gen_cache(name):
    return f"{RES}/gen_{name}.jsonl"
print(f"train={len(train_all)}  dev={len(dev)}  model={BASE_MODEL}  device={MD.device_info()}")

In [ ]:
# Free GPU memory held by models from earlier steps (matters when running everything in one session)
import gc, torch
for _name in ["model", "m7", "trainer", "m"]:
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory in use: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
run = load_json("main_run"); BEST_FMT = run["schema_format"]
tok = MD.load_tokenizer(BASE_MODEL)
model = MD.load_model(BASE_MODEL, load_4bit=False, adapter=run["adapter"], merge=True)   # merged fp16 = fast inference
ft = cached("dev_finetuned", lambda: EV.score(dev, MD.generate(
    model, tok, EV.make_chats(dev, SPIDER, BEST_FMT), batch_size=16, cache_path=gen_cache("dev_finetuned")), SPIDER))

## 1. Headline comparison

In [ ]:
res = {"zero-shot": pd.read_parquet(f"{RES}/dev_zero-shot.parquet")}
if os.path.exists(f"{RES}/dev_3-shot.parquet"):
    res["3-shot"] = pd.read_parquet(f"{RES}/dev_3-shot.parquet")
res["QLoRA fine-tuned"] = ft
table = EV.compare_table(res)
table.to_csv(f"{RES}/headline_table.csv"); table.round(4)

In [ ]:
sig = {f"fine-tuned vs {k}": EV.mcnemar(res[k].exec_match, ft.exec_match) for k in res if k != "QLoRA fine-tuned"}
pd.DataFrame(sig).T

### Same 300-question subset: zero-shot vs 3-shot vs 7B vs fine-tuned

In [ ]:
subset = {"1.5B zero-shot": res["zero-shot"].iloc[DEV_SUBSET]}
if os.path.exists(f"{RES}/subset_3shot.parquet"):
    subset["1.5B 3-shot"] = pd.read_parquet(f"{RES}/subset_3shot.parquet")
if os.path.exists(f"{RES}/subset_7b_zero_shot.parquet"):
    subset["7B zero-shot (4-bit)"] = pd.read_parquet(f"{RES}/subset_7b_zero_shot.parquet")
subset["1.5B QLoRA fine-tuned"] = ft.iloc[DEV_SUBSET]
ref = EV.compare_table(subset); ref.to_csv(f"{RES}/subset_comparison.csv"); display(ref.round(4))
if "1.5B 3-shot" in subset:
    sig["fine-tuned vs 3-shot (subset)"] = EV.mcnemar(subset["1.5B 3-shot"].exec_match.reset_index(drop=True),
                                                      subset["1.5B QLoRA fine-tuned"].exec_match.reset_index(drop=True))
pd.DataFrame(sig).T

In [ ]:
acc = table[[f"acc_{h}" for h in HARDNESS_ORDER]].T; acc.index = HARDNESS_ORDER
ax = acc.plot.bar(figsize=(8, 4), rot=0, color=["#BBBBBB", "#8172B3", "#C44E52"][-acc.shape[1]:])
ax.set_ylabel("execution accuracy"); ax.set_title("Spider dev: execution accuracy by hardness"); ax.set_ylim(0, 1)
for c in ax.containers: ax.bar_label(c, fmt="%.2f", fontsize=8)
savefig("03_accuracy_by_hardness")

## 2. Error taxonomy: what kinds of mistakes does fine-tuning fix?

In [ ]:
err = pd.DataFrame({k: v.error_category.value_counts() for k, v in res.items()}).fillna(0).astype(int)
err = err.loc[err.sum(axis=1).sort_values().index]
err.to_csv(f"{RES}/error_taxonomy.csv")
ax = err.plot.barh(figsize=(9, 5.5), color=["#BBBBBB", "#8172B3", "#C44E52"][-err.shape[1]:])
ax.set_xlabel("# failed dev questions"); ax.set_title("Error categories (lower is better)")
savefig("03_error_taxonomy"); err

In [ ]:
BASE_NAME = "3-shot" if "3-shot" in res else "zero-shot"
base = res[BASE_NAME]
fixed = ft[(~base.exec_match.values) & ft.exec_match.values]
broken = ft[(base.exec_match.values) & ~ft.exec_match.values]
print(f"fixed by fine-tuning: {len(fixed)}   broken by fine-tuning: {len(broken)}\n")
for title, df in [("FIXED", fixed), ("BROKEN", broken)]:
    print(f"======== {title} (examples) ========")
    for i in df.sample(min(4, len(df)), random_state=0).index:
        print(f"[{dev[i].db_id} | {ft.hardness[i]}] {dev[i].question}\n  gold : {ft.gold[i]}\n  {BASE_NAME}: {base.pred[i]}"
              f"\n  FT   : {ft.pred[i]}\n")

In [ ]:
# Accuracy vs. schema size: do larger databases hurt?
ft["n_tables"] = [schema_stats(db_path(SPIDER, d))["n_tables"] for d in ft.db_id]
bins = pd.cut(ft.n_tables, [0, 3, 5, 8, 100], labels=["1-3", "4-5", "6-8", "9+"])
by_size = pd.DataFrame({k: v.exec_match.groupby(bins.values).mean() for k, v in res.items()})
by_size.plot(marker="o", figsize=(6, 3.5)); plt.ylabel("execution accuracy"); plt.xlabel("tables in database")
savefig("03_accuracy_by_schema_size"); by_size.round(3)

## 3. Auto-generated results summary

In [ ]:
abl = pd.read_csv(f"{RES}/ablations.csv") if os.path.exists(f"{RES}/ablations.csv") else None
fmt = load_json("schema_format_ablation")
lines = ["# Results summary (auto-generated by notebook 03)", "",
 f"* Base model: {BASE_MODEL}; adapter: LoRA r={run['r']} on all linear layers, 4-bit NF4 base (QLoRA)",
 f"* Trainable parameters: {run['trainable_params']/1e6:.1f}M of {run['total_params']/1e6:.0f}M; trained "
 f"{run['train_minutes']:.0f} min on {run['gpu']} (peak {run['peak_gpu_gb']:.1f} GB)",
 f"* **Spider dev (1,034) execution accuracy: zero-shot {table.loc['zero-shot','exec_acc']:.1%} → fine-tuned "
 f"{table.loc['QLoRA fine-tuned','exec_acc']:.1%}**",
 "* 300-question subset: " + ", ".join(f"{k} {v:.1%}" for k, v in ref.exec_acc.items()),
 f"* Execution-error rate: {table.loc['zero-shot','exec_error_rate']:.1%} → {table.loc['QLoRA fine-tuned','exec_error_rate']:.1%}",
 "* McNemar p-values: " + ", ".join(f"{k}: {v['p_value']:.2e}" for k, v in sig.items()),
 "", "## Headline", table.round(4).to_markdown(),
 "", "## Schema-format ablation (zero-shot, 300-question subset)", pd.DataFrame(fmt["table"]).round(4).to_markdown(index=False),
 "", "## Error taxonomy", err.to_markdown(),
 "", "## By schema size", by_size.round(3).to_markdown()]
lines += ["", "## Same 300-question subset", ref.round(4).to_markdown()]
if abl is not None:
    lines += ["", "## Ablations (300-question subset)", abl.round(4).to_markdown(index=False)]
open(f"{PROJECT_DIR}/reports/results_summary.md", "w").write("\n".join(lines)); print("\n".join(lines[:8]))